# ECT Cosmology — Interactive

**Paper §9** | Valeriy Blagovidov, *Euclidean Condensate Theory* (2026)

## Hubble tension via running G_eff(z)

The φ-first closure gives a dynamical effective Newton constant. A phenomenological
parametrisation consistent with CMB+BAO constraints is:

$$G_{\rm eff}(z)=G_N(1+z)^{2\varepsilon},\quad\varepsilon\approx0.01\ (\text{phenomenological fit})$$

This shifts the inferred Hubble constant by $\Delta H_0\approx+3\,{\rm km/s/Mpc}$,
alleviating (but not fully resolving) the Hubble tension.

> **Honest note (Level B):** $\varepsilon$ is fitted phenomenologically, not derived from
> first principles in the current version of ECT. The full derivation of $G_{\rm eff}(z)$
> from the condensate profile is a next step.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.integrate import quad
from ipywidgets import interact, FloatSlider
%matplotlib inline

Om, OL = 0.315, 0.685
H0_Pl = 67.4; H0_loc = 73.0
km_Mpc = 3.0857e19; Gyr = 3.1557e16

def H_ect(z, H0, eps):
    return H0 * np.sqrt(Om*(1+z)**3 * (1+z)**(2*eps) + OL)

def age(H0, eps):
    return quad(lambda z: 1/((1+z)*H_ect(z,H0,eps)), 0, 30)[0] * km_Mpc / Gyr

def plot_cosmo(H0=67.4, eps=0.01):
    z  = np.linspace(0, 3, 300)
    z2 = np.linspace(0, 15, 300)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    H_L = H0_Pl * np.sqrt(Om*(1+z)**3 + OL)
    H_E = np.array([H_ect(zi, H0, eps) for zi in z])
    t_L = age(H0_Pl, 0); t_E = age(H0, eps)
    axes[0].plot(z, H_L, 'b-',  lw=2, label=f'ΛCDM  H₀={H0_Pl}')
    axes[0].plot(z, H_E, 'g--', lw=2, label=f'ECT  H₀={H0:.1f}, ε={eps:.3f}')
    axes[0].axhline(H0_loc, color='r', ls=':', lw=1.5, alpha=0.7, label=f'Local H₀={H0_loc}')
    axes[0].set_title(f'H(z) | Age: ΛCDM={t_L:.2f} Gyr, ECT={t_E:.2f} Gyr')
    axes[0].set_xlabel('z'); axes[0].set_ylabel('H [km/s/Mpc]')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].plot(z2, (1+z2)**(2*eps), 'g-', lw=2, label=f'ε={eps:.3f}')
    axes[1].axhline(1, color='b', ls='--', lw=1.5, label='ΛCDM (ε=0)')
    axes[1].fill_between(z2, 1, (1+z2)**(2*eps), alpha=0.2, color='g')
    axes[1].set_xlabel('z'); axes[1].set_ylabel(r'$G_{\rm eff}(z)/G_N$')
    axes[1].set_title('Running G_eff(z)  [phenomenological]')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    Z, Nu = np.meshgrid(np.linspace(4,16,50), np.linspace(2,7,50))
    F = (1+Z)**(2*eps)
    Enh = np.log10(np.clip(np.exp(Nu**2/2*(1-1/F)), 1, 1e6))
    im = axes[2].contourf(Z, Nu, Enh, levels=20, cmap='hot_r')
    plt.colorbar(im, ax=axes[2], label='log₁₀ halo count enhancement')
    axes[2].contour(Z, Nu, Enh, levels=[0.5,1,2,3], colors='white', linewidths=0.8)
    axes[2].set_xlabel('z'); axes[2].set_ylabel('ν (peak height)')
    axes[2].set_title('JWST: Press-Schechter halo enhancement')
    plt.tight_layout(); plt.show()
    dH = H0 - H0_Pl
    print(f'ΔH₀ = {dH:+.1f} km/s/Mpc  | Tension: {abs(H0_loc-H0):.1f} km/s/Mpc remaining')
    print(f'Δage = {t_E-t_L:+.3f} Gyr  |  G_eff(z=10) = {(11)**(2*eps):.4f}×G_N')

interact(plot_cosmo,
  H0  = FloatSlider(value=67.4, min=60, max=75, step=0.1, description='H₀:',
                    continuous_update=False, style={'description_width':'initial'}),
  eps = FloatSlider(value=0.01, min=0, max=0.05, step=0.001, description='ε:',
                    continuous_update=False, style={'description_width':'initial'}));

## Primordial perturbations — slow-roll comparison

ECT does not yet provide a full derivation of the primordial spectrum. For comparison,
slow-roll single-field inflation gives:

$$n_s = 1 - \frac{2}{N_e}, \quad r = \frac{8}{N_e}$$

Observed (Planck 2018 + BICEP/Keck): $n_s=0.9649\pm0.0042$, $r<0.056$.

> **ECT status (Level C / research programme):** condensate fluctuation variables
> $\delta n_A$, $\delta\sigma$, $\delta\theta$ are identified as natural candidates.
> The pre-Lorentzian stage may admit an approximately scale-free regime.
> Full derivation of $n_s$, $A_s$, $r$ is the next major step — see §9 of the paper.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def plot_inflation(Ne=60):
    ns = 1 - 2/Ne; r_val = 8/Ne
    Ne_range = np.arange(40, 80)
    ns_range = 1 - 2/Ne_range
    r_range  = 8/Ne_range

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(Ne_range, ns_range, 'b-', lw=2, label='Slow-roll: $n_s=1-2/N_e$')
    ax1.axhline(0.9649, color='red', ls='--', lw=1.5, label='Planck 2018: 0.9649±0.0042')
    ax1.fill_between(Ne_range, 0.9607, 0.9691, alpha=0.2, color='red')
    ax1.axvline(Ne, color='blue', ls=':', lw=2)
    ax1.scatter([Ne], [ns], c='blue', s=100, zorder=5, label=f'N_e={Ne}: n_s={ns:.4f}')
    ax1.set_xlabel('$N_e$ (e-folds)'); ax1.set_ylabel('$n_s$')
    ax1.set_title('Spectral index (slow-roll formula)')
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    ax2.plot(Ne_range, r_range, 'b-', lw=2, label='Slow-roll: r=8/N_e')
    ax2.axhline(0.056, color='red', ls='--', lw=1.5, label='BICEP/Keck 95% CL: r<0.056')
    ax2.axvline(Ne, color='blue', ls=':', lw=2)
    ax2.scatter([Ne], [r_val], c='blue', s=100, zorder=5, label=f'N_e={Ne}: r={r_val:.4f}')
    ax2.set_xlabel('$N_e$ (e-folds)'); ax2.set_ylabel('$r$')
    ax2.set_title('Tensor-to-scalar ratio')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    dev = abs(ns - 0.9649)/0.0042
    print(f'Slow-roll: n_s={ns:.4f}, r={r_val:.4f} | deviation from Planck: {dev:.1f}σ')
    print('ECT: candidate variables identified; full spectrum derivation in progress.')

interact(plot_inflation,
         Ne=IntSlider(value=60, min=40, max=80, step=1, description='N_e:',
                      style={'description_width':'initial'}));